# Demo Chapter 18: Context-Free Grammars and Constituency Parsing

Demo này minh hoạ các kiến thức chính của Chapter 18 bằng một notebook tự chạy được, không cần tải model lớn hay dataset ngoài.

Nội dung demo:
- **18.1 Constituency**: kiểm tra một nhóm từ có thể là constituent hay không.
- **18.2 Context-Free Grammar (CFG)**: biểu diễn grammar, lexicon, sinh câu và parse tree.
- **18.4 Chomsky Normal Form (CNF)**: vì sao CKY cần grammar dạng nhị phân.
- **18.5 Ambiguity**: một câu có thể có nhiều parse tree.
- **18.6 CKY Parsing**: tự implement CKY để parse câu.
- **18.7 Span-based Neural Constituency Parsing**: mô phỏng ý tưởng chấm điểm từng span.
- **18.8 Parser Evaluation**: tính labeled precision, recall, F1.
- **18.9 Head Finding**: tìm head word trong constituent.

Mục tiêu: cho thấy ta hiểu flow của chương: **CFG → Parse Tree → Ambiguity → CKY → Neural Span Score → Evaluation**.


# 18.1 Constituency: nhóm từ hoạt động như một đơn vị

Trong constituency parsing, ta không chỉ nhìn từng từ riêng lẻ mà nhìn các **cụm từ** như:
- NP = Noun Phrase
- VP = Verb Phrase
- PP = Prepositional Phrase

Ví dụ câu:

> I prefer a morning flight

Trong câu này, cụm **a morning flight** là một NP vì nó hoạt động như object của verb **prefer**.


In [ ]:
sentence = ["I", "prefer", "a", "morning", "flight"]

# Một vài span giả lập để minh hoạ constituent
candidate_spans = {
    (0, 1): "NP",        # I
    (2, 5): "NP",        # a morning flight
    (1, 5): "VP",        # prefer a morning flight
    (0, 5): "S",         # whole sentence
}

def show_spans(words, spans):
    print("Sentence:", " ".join(words))
    print("\nCandidate constituents:")
    for (i, j), label in spans.items():
        print(f"[{i},{j}] {label:>2} -> {' '.join(words[i:j])}")

show_spans(sentence, candidate_spans)

# 18.2 Context-Free Grammar (CFG)

Một CFG gồm các luật dạng:

```text
A → β
```

Trong đó:
- `A` là **non-terminal** như S, NP, VP.
- `β` là chuỗi terminal hoặc non-terminal.
- Terminal là từ thật như `I`, `flight`, `prefer`.

Ta sẽ tạo một grammar nhỏ dựa theo ví dụ trong chương.


In [ ]:
# Grammar đơn giản cho câu: I prefer a morning flight
grammar = {
    "S":  [["NP", "VP"]],
    "NP": [["Pronoun"], ["Det", "Nominal"]],
    "VP": [["Verb", "NP"]],
    "Nominal": [["Noun"], ["Nominal", "Noun"]],
    "Pronoun": [["I"]],
    "Det": [["a"]],
    "Verb": [["prefer"]],
    "Noun": [["morning"], ["flight"]],
}

for lhs, rhss in grammar.items():
    alternatives = [" ".join(rhs) for rhs in rhss]
    print(f"{lhs:8} -> {' | '.join(alternatives)}")

## Sinh câu từ CFG

CFG có thể được xem như một **generator**: bắt đầu từ `S`, liên tục rewrite bằng các luật cho đến khi chỉ còn terminal words.


In [ ]:
def is_nonterminal(symbol):
    return symbol in grammar

def generate(symbol, chosen_rules=None):
    """Generate một chuỗi từ grammar bằng cách chọn rule đầu tiên, hoặc chọn theo chosen_rules."""
    if not is_nonterminal(symbol):
        return [symbol]

    rules = grammar[symbol]
    rule_id = 0
    if chosen_rules and symbol in chosen_rules:
        rule_id = chosen_rules[symbol].pop(0)

    output = []
    for child in rules[rule_id]:
        output.extend(generate(child, chosen_rules))
    return output

# Chọn rule để sinh đúng câu "I prefer a morning flight"
chosen_rules = {
    "S": [0],
    "NP": [0, 1],        # NP đầu tiên -> Pronoun, NP thứ hai -> Det Nominal
    "VP": [0],
    "Nominal": [1, 0],   # Nominal -> Nominal Noun, sau đó Nominal -> Noun
    "Pronoun": [0],
    "Det": [0],
    "Verb": [0],
    "Noun": [0, 1],      # morning, flight
}

generated = generate("S", {k: v.copy() for k, v in chosen_rules.items()})
print("Generated sentence:", " ".join(generated))

## Parse tree

Một parse tree cho biết cấu trúc phân cấp của câu. Dưới đây ta biểu diễn tree bằng nested tuple:

```text
(label, child1, child2, ...)
```


In [ ]:
tree = (
    "S",
    ("NP", ("Pronoun", "I")),
    ("VP",
        ("Verb", "prefer"),
        ("NP",
            ("Det", "a"),
            ("Nominal",
                ("Nominal", ("Noun", "morning")),
                ("Noun", "flight")
            )
        )
    )
)

def pretty_tree(t, indent=0):
    if isinstance(t, str):
        print("  " * indent + t)
        return
    label = t[0]
    print("  " * indent + label)
    for child in t[1:]:
        pretty_tree(child, indent + 1)

pretty_tree(tree)

# 18.4 Chomsky Normal Form (CNF)

CKY cần grammar ở dạng CNF:

```text
A → B C
A → word
```

Tức là mỗi rule chỉ có:
- 2 non-terminal ở vế phải, hoặc
- 1 terminal ở vế phải.

Ví dụ rule không phải CNF:

```text
S → Aux NP VP
```

có 3 symbol bên phải, nên cần chuyển thành:

```text
S  → X1 VP
X1 → Aux NP
```


In [ ]:
def binarize_rule(lhs, rhs):
    """Binarize một rule dài hơn 2 symbol thành các rule nhị phân."""
    if len(rhs) <= 2:
        return [(lhs, rhs)]

    rules = []
    current_lhs = lhs
    remaining = rhs[:]
    counter = 1

    while len(remaining) > 2:
        new_symbol = f"{lhs}_BIN{counter}"
        rules.append((current_lhs, [remaining[0], new_symbol]))
        current_lhs = new_symbol
        remaining = remaining[1:]
        counter += 1

    rules.append((current_lhs, remaining))
    return rules

original_lhs = "S"
original_rhs = ["Aux", "NP", "VP"]

print("Original rule:")
print(f"{original_lhs} -> {' '.join(original_rhs)}")

print("\nAfter binarization:")
for lhs, rhs in binarize_rule(original_lhs, original_rhs):
    print(f"{lhs} -> {' '.join(rhs)}")

# 18.5 Ambiguity: một câu có nhiều parse tree

Ví dụ trong chương:

> Book the flight through Houston

Câu này có thể hiểu theo nhiều cách:
1. `through Houston` bổ nghĩa cho **flight** → chuyến bay đi qua Houston.
2. `through Houston` bổ nghĩa cho hành động **book** → đặt vé thông qua Houston.
3. Do grammar CNF có thể biểu diễn lại rule `VP → Verb NP PP`, ta cũng có thêm cách parse tương ứng với cấu trúc này.

Bây giờ ta sẽ implement CKY để thấy câu này có nhiều parse.


# 18.6 CKY Parsing

Ta dùng grammar CNF nhỏ gần giống L1 trong sách.

Ý tưởng CKY:
- Mỗi ô `[i,j]` trong bảng lưu các non-terminal có thể sinh ra span `words[i:j]`.
- Bắt đầu từ từng word.
- Sau đó ghép các span nhỏ thành span lớn bằng rule `A → B C`.
- Nếu ô `[0,n]` có `S`, câu parse được.


In [ ]:
from collections import defaultdict
from functools import lru_cache

# CNF grammar cho câu: Book the flight through Houston
# Dùng lowercase cho dễ xử lý.
lexical_rules = {
    "book":    ["Verb", "Noun", "Nominal", "VP", "S"],
    "the":     ["Det"],
    "flight":  ["Noun", "Nominal"],
    "through": ["Preposition"],
    "houston": ["ProperNoun", "NP"],
}

binary_rules = [
    ("S", "NP", "VP"),
    ("S", "X1", "VP"),
    ("X1", "Aux", "NP"),
    ("S", "Verb", "NP"),
    ("S", "X2", "PP"),
    ("S", "Verb", "PP"),
    ("S", "VP", "PP"),

    ("NP", "Det", "Nominal"),
    ("Nominal", "Nominal", "Noun"),
    ("Nominal", "Nominal", "PP"),

    ("VP", "Verb", "NP"),
    ("VP", "X2", "PP"),
    ("X2", "Verb", "NP"),
    ("VP", "Verb", "PP"),
    ("VP", "VP", "PP"),

    ("PP", "Preposition", "NP"),
]

# index binary rules theo cặp (B,C)
binary_index = defaultdict(list)
for A, B, C in binary_rules:
    binary_index[(B, C)].append(A)

words = "book the flight through houston".split()
n = len(words)

print("Words:", words)
print("Number of words:", n)

In [ ]:
def cky_parse(words):
    n = len(words)
    table = [[defaultdict(list) for _ in range(n + 1)] for _ in range(n + 1)]

    # Fill diagonal: A -> word
    for j in range(1, n + 1):
        word = words[j - 1]
        for A in lexical_rules.get(word, []):
            table[j - 1][j][A].append(("LEX", word))

        # Fill larger spans ending at j
        for i in range(j - 2, -1, -1):
            for k in range(i + 1, j):
                left_labels = list(table[i][k].keys())
                right_labels = list(table[k][j].keys())

                for B in left_labels:
                    for C in right_labels:
                        for A in binary_index.get((B, C), []):
                            table[i][j][A].append((k, B, C))

    return table

table = cky_parse(words)

# In bảng CKY
for span_len in range(1, n + 1):
    print(f"\nSpans of length {span_len}:")
    for i in range(0, n - span_len + 1):
        j = i + span_len
        labels = sorted(table[i][j].keys())
        if labels:
            print(f"[{i},{j}] {' '.join(words[i:j]):30} -> {labels}")

## Nhận diện câu có parse được không?

Nếu `S` nằm trong ô `[0,n]`, CKY nhận diện rằng câu này có ít nhất một parse hợp lệ.


In [ ]:
if "S" in table[0][n]:
    print("Sentence is grammatical under this CFG.")
    print("Number of S backpointers in [0,n]:", len(table[0][n]["S"]))
else:
    print("Sentence is NOT grammatical under this CFG.")

## CKY Parser: lấy lại toàn bộ parse tree

Recognizer chỉ trả lời được “có parse hay không”.

Parser cần thêm **backpointer** để reconstruct parse tree.


In [ ]:
@lru_cache(None)
def build_trees(i, j, label):
    entries = table[i][j][label]
    results = []

    for entry in entries:
        if entry[0] == "LEX":
            _, word = entry
            results.append((label, word))
        else:
            k, B, C = entry
            left_trees = build_trees(i, k, B)
            right_trees = build_trees(k, j, C)
            for lt in left_trees:
                for rt in right_trees:
                    results.append((label, lt, rt))

    return tuple(results)

all_parses = build_trees(0, n, "S")
print("Number of parses:", len(all_parses))

for idx, parse in enumerate(all_parses, start=1):
    print(f"\nParse tree {idx}")
    pretty_tree(parse)

# 18.7 Span-based Neural Constituency Parsing

Classical CKY dùng grammar để quyết định span nào hợp lệ.

Neural constituency parser hiện đại làm khác:
- Encoder như BERT/Transformer tạo representation cho từng word.
- Mỗi span `(i,j)` được chấm điểm cho từng label như NP, VP, PP, S.
- Sau đó dùng một biến thể CKY để tìm tree có tổng score cao nhất.

Ở đây ta không train model thật, mà **mô phỏng span score** để minh hoạ đúng ý tưởng của chương.


In [ ]:
import math
import matplotlib.pyplot as plt

# Mock score cho một số span quan trọng
# score[(i,j,label)] = neural score
score = defaultdict(lambda: -10.0)

# Leaves / short spans
score[(0, 1, "Verb")] = 2.0
score[(1, 2, "Det")] = 2.0
score[(2, 3, "Nominal")] = 2.0
score[(3, 4, "Preposition")] = 2.0
score[(4, 5, "NP")] = 2.0

# Larger spans
score[(1, 3, "NP")] = 5.0                         # the flight
score[(3, 5, "PP")] = 4.5                         # through Houston
score[(1, 5, "NP")] = 3.0                         # the flight through Houston
score[(0, 3, "VP")] = 4.0                         # book the flight
score[(0, 5, "S")] = 5.0                          # whole sentence

labels = ["NP", "VP", "PP", "S", "Nominal", "Verb", "Det", "Preposition"]

def best_label_for_span(i, j):
    candidates = [(label, score[(i, j, label)]) for label in labels]
    return max(candidates, key=lambda x: x[1])

important_spans = [(0,1), (1,3), (3,5), (1,5), (0,3), (0,5)]

for i, j in important_spans:
    label, sc = best_label_for_span(i, j)
    print(f"[{i},{j}] {' '.join(words[i:j]):30} -> best label = {label:10} score = {sc}")

In [ ]:
# Visualize một vài span score
span_names = [f"[{i},{j}] {' '.join(words[i:j])}" for i, j in important_spans]
span_scores = [best_label_for_span(i, j)[1] for i, j in important_spans]
span_labels = [best_label_for_span(i, j)[0] for i, j in important_spans]

plt.figure(figsize=(10, 4))
plt.bar(span_names, span_scores)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Best neural span score")
plt.title("Mock span scores for constituency parsing")
for x, y, label in zip(range(len(span_scores)), span_scores, span_labels):
    plt.text(x, y + 0.1, label, ha="center")
plt.tight_layout()
plt.show()

## Neural CKY: tìm binary tree có tổng score cao nhất

Công thức ý tưởng:

```text
best(i,j) = max_label score(i,j,label) + max_k [best(i,k) + best(k,j)]
```

Khác với CKY cổ điển:
- CKY cổ điển kiểm tra rule `A → B C`.
- Neural CKY chủ yếu dựa vào score học được từ data.


In [ ]:
def neural_cky(words):
    n = len(words)
    best = [[-math.inf for _ in range(n + 1)] for _ in range(n + 1)]
    back = [[None for _ in range(n + 1)] for _ in range(n + 1)]

    for length in range(1, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            best_label, label_score = best_label_for_span(i, j)

            if length == 1:
                best[i][j] = label_score
                back[i][j] = (best_label, None)
            else:
                best_split_score = -math.inf
                best_k = None
                for k in range(i + 1, j):
                    candidate = best[i][k] + best[k][j]
                    if candidate > best_split_score:
                        best_split_score = candidate
                        best_k = k
                best[i][j] = label_score + best_split_score
                back[i][j] = (best_label, best_k)

    return best, back

best, back = neural_cky(words)
print("Best whole-tree score:", best[0][n])
print("Root decision:", back[0][n])

In [ ]:
def build_neural_tree(i, j):
    label, k = back[i][j]
    if k is None:
        return (label, words[i])
    return (label, build_neural_tree(i, k), build_neural_tree(k, j))

neural_tree = build_neural_tree(0, n)
pretty_tree(neural_tree)

# 18.8 Evaluating Parsers: PARSEVAL

Parser thường được đánh giá bằng:
- **Labeled Precision**
- **Labeled Recall**
- **F1**

Một constituent đúng nếu:
- cùng start index
- cùng end index
- cùng label


In [ ]:
def collect_constituents(t, start=0):
    """Return set of (start, end, label), bỏ qua preterminal -> word nếu muốn đơn giản."""
    label = t[0]

    # leaf: (Label, word)
    if len(t) == 2 and isinstance(t[1], str):
        return set(), start + 1

    constituents = set()
    cur = start
    for child in t[1:]:
        child_constituents, cur = collect_constituents(child, cur)
        constituents |= child_constituents

    end = cur
    constituents.add((start, end, label))
    return constituents, end

# Gold tree: chọn parse đầu tiên từ CFG làm reference
gold_tree = all_parses[0]

# Hypothesis tree: neural tree
gold_const, _ = collect_constituents(gold_tree)
hyp_const, _ = collect_constituents(neural_tree)

correct = gold_const & hyp_const
precision = len(correct) / len(hyp_const) if hyp_const else 0
recall = len(correct) / len(gold_const) if gold_const else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

print("Gold constituents:")
print(sorted(gold_const))
print("\nHypothesis constituents:")
print(sorted(hyp_const))
print("\nCorrect constituents:")
print(sorted(correct))

print(f"\nLabeled Precision = {precision:.3f}")
print(f"Labeled Recall    = {recall:.3f}")
print(f"F1                = {f1:.3f}")

# 18.9 Heads and Head Finding

Mỗi constituent thường có một **head word**:
- NP: head thường là noun chính.
- VP: head thường là verb.
- PP: head thường là preposition hoặc object tuỳ hệ thống.

Ví dụ:
- `the flight` có head là `flight`.
- `book the flight` có head là `book`.

Dưới đây là một bộ rule đơn giản để minh hoạ head finding.


In [ ]:
def head_word(t):
    label = t[0]

    if len(t) == 2 and isinstance(t[1], str):
        return t[1]

    children = t[1:]

    # Rule đơn giản để demo
    if label == "NP":
        # ưu tiên head bên phải, thường là noun/proper noun
        return head_word(children[-1])
    if label == "VP":
        # ưu tiên verb bên trái
        return head_word(children[0])
    if label == "PP":
        # trong nhiều dependency convention, preposition có thể là head
        return head_word(children[0])
    if label == "S":
        # head của sentence thường lấy theo VP
        return head_word(children[-1])

    return head_word(children[-1])

print("Gold parse tree:")
pretty_tree(gold_tree)

print("\nHead word of each top-level constituent:")
for child in gold_tree[1:]:
    print(f"{child[0]} -> {head_word(child)}")

print("\nHead word of whole sentence:", head_word(gold_tree))

# Kết luận demo

Notebook này đã minh hoạ các nội dung chính của Chapter 18:

1. **Constituency**: nhóm từ có thể hoạt động như một đơn vị.
2. **CFG**: dùng rule để mô hình hoá cấu trúc câu.
3. **Parse Tree**: biểu diễn cấu trúc cú pháp dạng cây.
4. **CNF**: chuẩn hoá grammar thành dạng nhị phân để dùng CKY.
5. **Ambiguity**: một câu có thể có nhiều parse tree.
6. **CKY**: dynamic programming để tìm tất cả parse hợp lệ.
7. **Neural Span Parser**: chấm điểm từng span rồi tìm tree tốt nhất.
8. **PARSEVAL**: đánh giá parser bằng precision, recall, F1.
9. **Head Finding**: tìm từ trung tâm trong constituent.

Điểm quan trọng nhất:  
**Chapter 18 chuyển từ việc gán nhãn từng từ riêng lẻ sang việc phân tích cấu trúc phân cấp của cả câu.**
